In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Agent Platform での Gemini グラウンディング入門 (v2)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Colab で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fkazunori279%2Fgcp-blogs%2Fmain%2F20260602_gemini_grounding%2Fintro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Colab Enterprise で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/kazunori279/gcp-blogs/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Workbench で開く
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> GitHub で表示
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>共有:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2-ja.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| 著者 |
| --- |
| [Kazunori Sato](https://github.com/kazunori279) |

## 概要

[Agent Platform の Gemini グラウンディング](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/overview)を使用すると、生成テキストモデルで独自のドキュメントやデータに基づいたコンテンツを生成できます。この機能により、モデルは実行時にトレーニングデータを超えた情報にアクセスできます。モデルの応答を Google 検索結果や独自データでグラウンディングすることで、より正確で最新かつ関連性の高い応答を生成できます。

このノートブックでは、[Concierge AI](https://grounding-demo-761793285222.us-east1.run.app/) デモアプリで使用されている **6 つのグラウンディング手法**を紹介します。Concierge AI は [Gemini Live API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/live-api) を使った音声エージェントで、高級ホテルのコンシェルジュとして、[Google 検索](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-search)、[Google マップ](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-maps)、[Vector Search 2.0](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview) のグラウンディングを活用します。

| # | グラウンディングの種類 | API / 機能 | Concierge ツール |
|---|---|---|---|
| 1 | Google 検索 | 組み込み `GoogleSearch` ツール | `web_search`（テキスト） |
| 2 | Google マップ | `GoogleMaps` ツール（場所検索、ルート、ルート沿い検索） | `web_search`（マップ）、`maps_route`、`maps_search_along_route` |
| 3 | **Vector Search 2.0** | `vectorsearch_v1beta` SDK（自動エンベディングによるセマンティック検索） | `vs2_search` |
| 4 | 画像検索 | `enhancedContent.imageSearch`（REST） | `web_search`（画像） |
| 5 | 動画検索 | 動画寄りクエリによる Google 検索 | `web_search`（動画） |
| 6 | 画像生成のための画像検索 | `searchTypes.imageSearch`（REST） | `request_postcard` → `illustrate_place` |

### 目標

このチュートリアルでは、以下の方法を学びます：

- Google 検索結果に基づいた LLM テキストの生成
- Google マップデータ（場所検索、ルート、ルート沿い検索）に基づいた LLM テキストの生成
- 自動エンベディング付き Vector Search 2.0 コレクションの作成とセマンティック検索の実行
- `enhancedContent.imageSearch` による画像検索
- Google 検索グラウンディングによる YouTube 動画の検索
- `searchTypes.imageSearch` による Google 画像検索結果に基づいた画像生成

このチュートリアルでは、以下の Google Cloud AI サービスとリソースを使用します：

- Agent Platform（[Gemini API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models)）
- Agent Platform Vector Search 2.0

実行する手順：

- 各種サンプル用の LLM とプロンプトの設定
- Google 検索および Google マップのグラウンディング付きプロンプトの送信
- ホテルナレッジベース用の自動エンベディング付き Vector Search 2.0 コレクションのセットアップ
- REST API による画像・動画の検索
- Google 画像検索結果に基づいた画像の生成

## 始める前に

### Google Cloud プロジェクトの設定

**以下の手順は、ノートブックの環境に関係なく必要です。**

1. [Google Cloud プロジェクトを選択または作成](https://console.cloud.google.com/cloud-resource-manager)します。初回アカウント作成時には、コンピューティング/ストレージ費用に使える $300 の無料クレジットが付与されます。
1. [プロジェクトの課金が有効になっていること](https://cloud.google.com/billing/docs/how-to/modify-project)を確認します。
1. [Agent Platform、Vector Search API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com,vectorsearch.googleapis.com) を有効にします。
1. このノートブックをローカルで実行する場合は、[Cloud SDK](https://cloud.google.com/sdk) のインストールが必要です。

### [Google Gen AI SDK](https://googleapis.github.io/python-genai/)、[ADK](https://google.github.io/adk-docs/)、Vector Search SDK のインストール

このノートブックの実行に必要なパッケージをインストールします。

In [ ]:
# noqa: E225
%pip install --upgrade --quiet \
    google-adk \
    google-cloud-vectorsearch \
    google-auth \
    requests

### Google Cloud アカウントの認証

このノートブックを Google Colab で実行している場合は、環境の認証が必要です。以下のセルを実行してください。Colab Enterprise または Agent Platform Workbench を使用している場合、この手順は不要です。

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

### Google Cloud プロジェクト情報の設定とクライアントの作成

Agent Platform を使い始めるには、既存の Google Cloud プロジェクトが必要で、[Agent Platform API を有効にする](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)必要があります。

[プロジェクトと開発環境の設定](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/start)について詳しくはこちらをご覧ください。

**プロジェクト ID がわからない場合**は、以下をお試しください：
* `gcloud config list` を実行
* `gcloud projects list` を実行
* サポートページ：[プロジェクト ID の確認](https://support.google.com/googleapi/answer/7014113)

Agent Platform で使用する `LOCATION` 変数を変更することもできます。[Agent Platform のリージョン](https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations)について詳しくはこちらをご覧ください。

In [ ]:
import os  # noqa: E402

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

from google import genai  # noqa: E402

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=LOCATION
)

### ライブラリのインポート

グラウンディング設定に使用する Google Gen AI SDK の型と、REST API 呼び出しおよび表示用のユーティリティをインポートします。

In [ ]:
import base64
import json
import re

import google.auth
import google.auth.transport.requests as auth_requests
import requests as http_requests
from IPython.core.display import Markdown
from IPython.display import Image as IPImage
from IPython.display import display
from google.genai.types import (
    GenerateContentConfig,
    GenerateContentResponse,
    GoogleMaps,
    GoogleSearch,
    LatLng,
    Part,
    Tool,
    ToolConfig,
    RetrievalConfig,
)

### ヘルパー関数

`print_grounding_data()` はグラウンディングされた応答をインライン引用マーカーとソースリンク付きでレンダリングします。`rest_endpoint()` は SDK 未対応の機能向けに Gemini REST URL を構築します。

In [ ]:
def print_grounding_data(response: GenerateContentResponse):
    """Print response with grounding citations in Markdown."""
    candidate = (
        response.candidates[0] if response.candidates else None
    )
    metadata = getattr(candidate, "grounding_metadata", None)

    if not metadata:
        print("Response does not contain grounding metadata.")
        display(Markdown(response.text))
        return

    ENCODING = "utf-8"
    text_bytes = response.text.encode(ENCODING)
    parts = []
    last = 0

    for support in metadata.grounding_supports or []:
        end = support.segment.end_index
        parts.append(text_bytes[last:end].decode(ENCODING))
        indices = support.grounding_chunk_indices
        parts.append(
            " " + "".join(f"[{i + 1}]" for i in indices)
        )
        last = end

    parts.append(text_bytes[last:].decode(ENCODING))
    parts.append("\n\n----\n## Grounding Sources\n")

    if chunks := metadata.grounding_chunks:
        parts.append("### Grounding Chunks\n")
        for i, chunk in enumerate(chunks, 1):
            ctx = (
                chunk.web
                or chunk.retrieved_context
                or chunk.maps
            )
            if not ctx:
                continue
            uri = (ctx.uri or "").replace(
                "gs://",
                "https://storage.googleapis.com/",
                1,
            ).replace(" ", "%20")
            title = ctx.title or "Source"
            parts.append(f"{i}. [{title}]({uri})\n")
            if getattr(ctx, "place_id", None):
                pid = ctx.place_id
                parts.append(
                    f"    - Place ID: `{pid}`\n\n"
                )
            if getattr(ctx, "text", None):
                parts.append(f"{ctx.text}\n\n")

    if metadata.web_search_queries:
        queries = metadata.web_search_queries
        parts.append(f"\n**Web Search Queries:** {queries}\n")
        if metadata.search_entry_point:
            sep = metadata.search_entry_point
            parts.append(
                f"\n**Search Entry Point:**\n"
                f"{sep.rendered_content}\n"
            )
    elif metadata.retrieval_queries:
        queries = metadata.retrieval_queries
        parts.append(
            f"\n**Retrieval Queries:** {queries}\n"
        )

    widget_attr = "google_maps_widget_context_token"
    if hasattr(metadata, widget_attr):
        token = getattr(metadata, widget_attr)
        if token:
            parts.append(
                f"\n**Maps Widget Token:**"
                f" `{token[:50]}...`\n"
            )

    display(Markdown("".join(parts)))


def get_authorized_session():
    """Create an authorized HTTP session for REST."""
    creds = google.auth.default()[0]
    return auth_requests.AuthorizedSession(creds)


def rest_endpoint(model: str) -> str:
    """Build the Agent Platform generateContent REST URL."""
    return (
        "https://aiplatform.googleapis.com/v1"
        f"/projects/{PROJECT_ID}"
        "/locations/global/publishers/google"
        f"/models/{model}:generateContent"
    )

Agent Platform から Gemini モデルを初期化します：

In [ ]:
MODEL_ID = "gemini-2.5-flash-lite"  # @param {type: "string"}
AGENT_MODEL_ID = "gemini-2.5-flash"  # @param {type: "string"}

---
## 1. Google 検索によるグラウンディング

Google 検索グラウンディングにより、Gemini は生成中にウェブ検索を実行し、その結果に基づいて回答を生成できます。レスポンスには**グラウンディングメタデータ**としてソース引用が含まれます：`grounding_chunks`（取得されたウェブページ）、`grounding_supports`（テキストの各部分と特定のソースとの対応と信頼度スコア）、`web_search_queries`（モデルが使用した実際の検索クエリ）。利用規約に従って表示する `search_entry_point` ウィジェットも返されます。

Concierge AI アプリでは、`web_search` ツールがこの機能を使用して、最新のイベント情報、スケジュール、チケット価格などの質問に回答します。

### グラウンディングなし

まず、グラウンディングなしで Gemini に時間に敏感な質問をするとどうなるか見てみましょう。モデルはトレーニングデータのみに依存するため、情報が古かったり不完全だったりする場合があります。

In [ ]:
PROMPT = "今週ラスベガスで上演中のショーは何ですか？"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
)

display(Markdown(response.text))

### Google 検索グラウンディングあり

`GoogleSearch` をツールとして追加することで、ソース引用付きのグラウンディングされた最新の回答を取得できます。モデルが自律的に検索クエリを決定・実行し、結果をインライン参照付きで回答に織り込みます。

In [ ]:
google_search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(tools=[google_search_tool]),
)

print_grounding_data(response)

---
## 2. Google マップによるグラウンディング

Google マップのグラウンディングは、Gemini を Google Maps Platform のデータに接続し、位置情報ベースのクエリに対応します。モデルはプロンプト内の地理的意図に基づいて、自律的に Maps API を呼び出すかどうかを判断します。`enable_widget=True` を設定するとテキスト応答とともにインタラクティブな埋め込みマップが返され、`RetrievalConfig` の `lat_lng` で特定のエリアに結果をバイアスできます。グラウンディングメタデータには、他の Google Maps Platform API で使用できる `place_id` が含まれます。

Concierge AI アプリでは、3 つのツールがこの機能を使用します：
- `web_search` — 周辺の場所を検索し、マップウィジェットを返す
- `maps_route` — 経路、距離、所要時間を取得
- `maps_search_along_route` — ドライブルート沿いの立ち寄りスポットを検索

### 場所検索

`GoogleMaps` ツールに自然言語クエリを渡して、周辺の場所を検索します。モデルはインタラクティブなマップウィジェット付きの場所検索結果を返します。

In [ ]:
google_maps_tool = Tool(google_maps=GoogleMaps(enable_widget=True))

PROMPT = "ラスベガス・ストリップ周辺のおすすめ寿司レストランを探して"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[google_maps_tool],
        tool_config=ToolConfig(
            retrieval_config=RetrievalConfig(
                # ラスベガス・ストリップを基準に「周辺」クエリを処理
                lat_lng=LatLng(latitude=36.1699, longitude=-115.1398)
            ),
        ),
    ),
)

print_grounding_data(response)

### ルート — 経路案内、距離、所要時間

これは `maps_route` ツールで使用されるパターンです。Google マップのグラウンディングが有効な場合、モデルは自然言語のプロンプトからルーティングの意図を推測します。

In [ ]:
from urllib.parse import quote_plus

ORIGIN = "3950 S Las Vegas Blvd, Las Vegas, NV 89119"
DESTINATION = "Red Rock Canyon"

PROMPT = (
    f"{ORIGIN} から"
    f"{DESTINATION}までの車での行き方を教えてください。"
    f"合計の所要時間と距離も含めてください。"
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

maps_url = (
    "https://www.google.com/maps/dir/?api=1"
    f"&origin={quote_plus(ORIGIN)}"
    f"&destination={quote_plus(DESTINATION)}"
    "&travelmode=driving"
)
display(Markdown(
    f"**[Google マップでルートを開く]({maps_url})**"
))

### ルート沿い検索 — ドライブルート沿いの POI

これは `maps_search_along_route` ツールで使用されるパターンです。モデルがルートを計算し、ルート沿いの該当する POI を検索します。

In [ ]:
ORIGIN = "3950 S Las Vegas Blvd, Las Vegas, NV 89119"
DESTINATION = "McCarran Airport"

PROMPT = (
    f"{ORIGIN} から"
    f"{DESTINATION}までのドライブルート沿いにある "
    "おすすめのカフェを3つ探してください。"
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

# グラウンディングチャンクから場所名と place_id を抽出
candidate = response.candidates[0] if response.candidates else None
gm = getattr(candidate, "grounding_metadata", None)
wp_names, wp_ids = [], []
if gm and gm.grounding_chunks:
    for chunk in gm.grounding_chunks:
        if not chunk.maps:
            continue
        if chunk.maps.title:
            wp_names.append(chunk.maps.title)
        pid = getattr(chunk.maps, "place_id", None)
        if pid:
            wp_ids.append(pid.removeprefix("places/"))

# Maps URLs API は waypoints と waypoint_place_ids を別パラメータで指定
maps_url = (
    "https://www.google.com/maps/dir/?api=1"
    f"&origin={quote_plus(ORIGIN)}"
    f"&destination={quote_plus(DESTINATION)}"
    "&travelmode=driving"
)
if wp_names:
    maps_url += "&waypoints=" + quote_plus(
        "|".join(wp_names)
    )
if wp_ids:
    maps_url += "&waypoint_place_ids=" + "|".join(wp_ids)

display(Markdown(
    f"**[Google マップで経由地付きルートを開く]({maps_url})**"
))

---
## 3. Vector Search 2.0 によるグラウンディング

Vector Search 2.0 は、フルマネージドの AI ネイティブ検索エンジンです。従来のインデックス・アズ・ア・サービスから進化し、包括的なストレージ＆検索システムになりました。インデックスを主要リソースとして管理する代わりに、**コレクション**と**データオブジェクト**を使って作業します。レプリケートされたスケーラブルなストレージエンジンが、AI アプリケーションの統合データソースとして機能します。

主要な機能は**自動エンベディング**です。ベクトルスキーマの `vertex_embedding_config` を通じて [`gemini-embedding-2`](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/embeddings) がエンベディングを自動生成します。`text_template` フィールドでエンベディング対象のテキストを制御し（例：`"{title} {category} {content}"`）、`task_type` でユースケースに応じた最適化を行います — インデックス時は `RETRIEVAL_DOCUMENT`、クエリ時は `QUESTION_ANSWERING` を使用します。

Concierge AI アプリでは、`vs2_search` ツールがホテルのナレッジベースコレクションにクエリを送信し、客室、ダイニング、スパ、アクセス、ポリシーなどに関するゲストの質問に回答します。

このセクションでは、小規模なデモコレクションを作成し、ホテル情報のアイテムを追加して、セマンティック検索クエリを実行します。

### Vector Search API の有効化

このセクションを使用する前に、プロジェクトで Vector Search API を有効にする必要があります。

In [ ]:
# noqa: E225
!gcloud services enable vectorsearch.googleapis.com --project={PROJECT_ID}

### 自動エンベディング付きコレクションの作成

データスキーマとベクトルスキーマを定義し、コレクションを作成します。ベクトルスキーマの `vertex_embedding_config` で、使用するエンベディングモデルとエンベディング用のテキスト入力の構成方法を指定します。

In [ ]:
from google.cloud import vectorsearch_v1beta

VS2_LOCATION = "us-central1"
COLLECTION_ID = "notebook-hotel-demo"  # @param {type: "string"}
VS2_PARENT = f"projects/{PROJECT_ID}/locations/{VS2_LOCATION}"
COLLECTION_PATH = f"{VS2_PARENT}/collections/{COLLECTION_ID}"

vs_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

# Data schema: fields stored alongside each vector
DATA_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "content": {"type": "string"},
        "category": {"type": "string"},
    },
}

# Vector schema: auto-embedding config — VS2 calls gemini-embedding-2
# automatically using the text_template to build the input text
VECTOR_SCHEMA = {
    "content_embedding": {
        "dense_vector": {
            "dimensions": 768,
            "vertex_embedding_config": {
                "model_id": "gemini-embedding-2",
                "text_template": "{title} {category} {content}",
                "task_type": "RETRIEVAL_DOCUMENT",
            },
        },
    },
}

# Create the collection (idempotent — skips if it already exists)
try:
    vs_client.get_collection(
        request=vectorsearch_v1beta.GetCollectionRequest(name=COLLECTION_PATH)
    )
    print(f"Collection '{COLLECTION_ID}' already exists")
except Exception:
    print(f"Creating collection '{COLLECTION_ID}'...")
    op = vs_client.create_collection(
        request=vectorsearch_v1beta.CreateCollectionRequest(
            parent=VS2_PARENT,
            collection_id=COLLECTION_ID,
            collection={
                "data_schema": DATA_SCHEMA,
                "vector_schema": VECTOR_SCHEMA,
            },
        )
    )
    op.result()
    print(f"Collection '{COLLECTION_ID}' created")

### ホテルナレッジベースアイテムの追加

各アイテムには `title`、`content`、`category` があります。`vectors` フィールドは空のままにします。Vector Search 2.0 がスキーマで定義された `vertex_embedding_config` を使用して、エンベディングを自動生成します。

In [ ]:
HOTEL_INFO = [
    {
        "id": "room-01",
        "title": "デラックスルーム",
        "content": (
            "45平米のお部屋。キングベッドまたはツインベッド、"
            "皇居の庭園を望む床から天井までの窓を備えています。"
            "シモンズ製ビューティレストマットレス、エジプト綿"
            "400スレッドカウントリネン、55インチ有機ELテレビ、"
            "ネスプレッソマシン、ミニバー、レインシャワーと"
            "深めの浴槽付き大理石バスルーム、温水洗浄便座、"
            "イソップのアメニティを完備。"
            "料金：1泊450ドルより。"
        ),
        "category": "客室",
    },
    {
        "id": "room-02",
        "title": "プレミアスイート",
        "content": (
            "85平米のスイート。独立したリビングエリア、"
            "4人掛けダイニングテーブル、東京のパノラマ"
            "スカイラインビューを備えています。デラックスルームの"
            "全アメニティに加え、Bang & Olufsen サウンドシステム、"
            "ウォークインクローゼット、ダブル洗面台バスルーム、"
            "エグゼクティブラウンジへのアクセス。ターンダウン"
            "サービスには和菓子をご用意。"
            "料金：1泊1,000ドルより。"
        ),
        "category": "客室",
    },
    {
        "id": "dine-01",
        "title": "レストラン 桜",
        "content": (
            "36階に位置するミシュラン2つ星の懐石料理レストラン。"
            "山本武シェフが手がける、築地直送の海鮮、和牛、"
            "京野菜を使った季節のおまかせコース。"
            "おまかせディナー：250ドル。"
            "営業時間：17:30〜21:30（ラストオーダー）。"
            "要予約。スマートカジュアルのドレスコード。"
        ),
        "category": "ダイニング",
    },
    {
        "id": "dine-02",
        "title": "ブラッスリー — オールデイダイニング",
        "content": (
            "ロビー階のインターナショナルレストラン。"
            "朝食ブッフェ（6:30〜10:30、40ドル）、"
            "ランチアラカルト（11:30〜14:30）、"
            "ディナー（17:30〜22:00）。"
            "朝食の新鮮な刺身ステーション、手打ちパスタ、"
            "日曜シャンパンブランチ（85ドル、"
            "ヴーヴ・クリコ飲み放題）が人気。"
            "お子様メニューあり。"
        ),
        "category": "ダイニング",
    },
    {
        "id": "spa-01",
        "title": "スパのご案内とトリートメント",
        "content": (
            "セレニティスパは5階全フロアを占め、"
            "12室のトリートメントルーム（カップルスイート3室含む）"
            "を備えています。シグネチャーメニュー：「桜リニューアル」"
            "— 桜オイルを使った90分のホットストーンマッサージ"
            "（195ドル）。その他の人気メニュー：指圧マッサージ"
            "（60分、125ドル）、檜フェイシャル（75分、155ドル）、"
            "抹茶ボディスクラブ（45分、100ドル）。"
            "営業時間：9:00〜21:00。"
        ),
        "category": "スパ",
    },
    {
        "id": "pol-01",
        "title": "チェックイン・チェックアウト時間",
        "content": (
            "チェックイン：15:00。チェックアウト：12:00（正午）。"
            "アーリーチェックイン（13:00〜）：50ドル"
            "（空室状況による）。レイトチェックアウト（15:00まで）"
            "：55ドル、18:00まで：半日料金。"
            "チェックイン前・チェックアウト後の手荷物預かりは"
            "無料です。"
        ),
        "category": "ポリシー",
    },
    {
        "id": "dir-01",
        "title": "成田空港からのアクセス",
        "content": (
            "成田国際空港から成田エクスプレス（N'EX）で東京駅まで"
            "約60分、丸ノ内線に乗り換えて大手町駅まで2分。"
            "C6b出口がホテルロビーに直結しています。"
            "また、リムジンバスが30分間隔で運行しています。"
            "専用車送迎：250ドル。"
        ),
        "category": "アクセス",
    },
    {
        "id": "fit-01",
        "title": "フィットネスセンター",
        "content": (
            "5階の24時間フィットネスセンター。"
            "テクノジム製設備：有酸素マシン15台（トレッドミル、"
            "バイク、エリプティカル、ローイングマシン）、"
            "フリーウェイトエリア（ダンベル50kgまで、"
            "スクワットラック、ベンチプレス）、TRXサスペンション"
            "トレーニング、ヨガマット付きストレッチエリア。"
            "パーソナルトレーニング：1時間85ドル。"
            "ウェアとタオルは無料でご利用いただけます。"
        ),
        "category": "フィットネス",
    },
    {
        "id": "act-01",
        "title": "ホテルでの文化体験",
        "content": (
            "宿泊者向けの無料文化アクティビティ（毎日開催）："
            "禅庭での朝の瞑想（6:30）、折り紙ワークショップ"
            "（10:00、ロビーラウンジ）、書道教室"
            "（14:00、カルチャールーム）、生け花"
            "（16:00、土曜日）。有料体験：茶道の先生による"
            "プライベート茶道体験（1名100ドル、60分）、"
            "ソムリエによる日本酒テイスティング（1名85ドル）。"
        ),
        "category": "アクティビティ",
    },
    {
        "id": "wifi-01",
        "title": "Wi-Fi・インターネット接続",
        "content": (
            "ホテル全館（ロビー、客室、レストラン、プール、スパ）"
            "で高速Wi-Fiを無料でご利用いただけます。"
            "ネットワーク名：'GrandSapphire-Guest'。"
            "接続時に部屋番号とお名前を入力してください。"
            "速度：下り200Mbps／上り100Mbps。"
            "全客室に有線LAN接続あり。"
            "ITサポート：内線9番（24時間対応）。"
        ),
        "category": "Wi-Fi",
    },
]

# vectors は空のまま — VS2 が vertex_embedding_config を使って自動生成
batch_request = [
    {
        "data_object_id": item["id"],
        "data_object": {
            "data": {
                "title": item["title"],
                "content": item["content"],
                "category": item["category"],
            },
            "vectors": {},
        },
    }
    for item in HOTEL_INFO
]

try:
    data_client.batch_create_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchCreateDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=batch_request,
            )
        )
    )
    print(f"{len(HOTEL_INFO)} 件のデータオブジェクトを作成しました")
except Exception as e:
    if "already exists" in str(e).lower():
        print("データオブジェクトは既に存在します（スキップ）")
    else:
        raise

### セマンティック検索

自然言語クエリを使用してホテルナレッジベースを検索します。`task_type="QUESTION_ANSWERING"` パラメータにより、エンベディングモデルに質問応答検索用の最適化を指示します。

In [ ]:
import time

# 自動エンベディングの生成を待機
time.sleep(3)


def vs2_search(query, category="", top_k=3):
    """ホテルナレッジベースを検索する。"""
    filter_kwargs = {}
    if category:
        filter_kwargs["filter"] = {
            "category": {"$eq": category}
        }

    request = vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COLLECTION_PATH,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="content_embedding",
            # QUESTION_ANSWERING はクエリ側のエンベディングを最適化
            # （インデックス時に使う RETRIEVAL_DOCUMENT とは異なる）
            task_type="QUESTION_ANSWERING",
            top_k=top_k,
            output_fields=(
                vectorsearch_v1beta.OutputFields(
                    data_fields=["*"],
                )
            ),
            **filter_kwargs,
        ),
    )
    results = search_client.search_data_objects(request)

    parts = [f"### 検索結果: *{query}*\n"]
    if category:
        parts.append(
            f"**フィルター:** category = `{category}`\n\n"
        )
    parts.append("\n")

    count = 0
    for item in results:
        data = item.data_object.data
        title = data.get("title", "")
        cat = data.get("category", "")
        content = data.get("content", "")[:200]
        parts.append(f"**{title}** (`{cat}`)<br>\n")
        parts.append(f"{content}...\n\n")
        count += 1

    if count == 0:
        parts.append("*結果が見つかりませんでした。*\n")

    display(Markdown("".join(parts)))


vs2_search("チェックアウトは何時ですか？")

In [ ]:
vs2_search("空港からホテルへの行き方を教えてください")

### フィルター検索

Vector Search 2.0 はベクトル類似度とペイロードフィルタリングを単一のクエリで組み合わせることができます。MongoDB スタイルの `$eq` フィルター演算子を使用して、カテゴリで結果を絞り込みます。

In [ ]:
vs2_search("どんな食事がありますか？", category="ダイニング")

In [ ]:
vs2_search("どんなアクティビティがありますか？", category="アクティビティ")

---
## 4. 画像検索

`enhancedContent.imageSearch` による画像検索は、Google 検索結果から画像のサムネイルを返します。Concierge AI アプリでは、`web_search` ツールがこの機能を使用して、場所、観光スポット、レストランの写真を表示します。

**注意:** これはプレビュー機能であり、プロジェクトのアローリストへの登録が必要です。プロジェクトで有効になっていない場合、呼び出しは成功しますが画像添付は 0 件となり、通常の Google 検索グラウンディングにフォールバックします。SDK がまだ Agent Platform での `enhancedContent.imageSearch` をサポートしていないため、REST API を使用します。

In [ ]:
# REST API — SDK はまだ enhancedContent.imageSearch に未対応
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": "夜のラスベガス・ストリップの写真",
        }],
    }],
    "tools": [{
        "googleSearch": {
            "enhancedContent": {"imageSearch": {}},
        },
    }],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

# 画像サムネイルは attachments フィールドに格納される
parts = ["### 画像検索結果\n\n"]
for att in gm.get("attachments", [])[:5]:
    img = att.get("image", {})
    if not img:
        continue
    source = img.get("source", {})
    thumbnail = img.get("thumbnail", {})
    title = source.get("title", "Untitled")
    uri = source.get("uri", "")
    parts.append(f"**{title}**\n")
    parts.append(f"- ソースページ: {uri}\n")
    thumb_uri = thumbnail.get("uri")
    if thumb_uri:
        parts.append(f"- サムネイル: ![]({thumb_uri})\n")
    parts.append("\n")

display(Markdown("".join(parts)))

---
## 5. 動画検索（YouTube）

動画検索は、動画寄りのクエリで Google 検索グラウンディングを使用して YouTube コンテンツを検索します。Concierge AI アプリでは、`web_search` ツールがテキスト、画像、マップの検索と並列でこれを実行します。

YouTube の動画 ID は、リダイレクト URL をたどることでグラウンディングチャンクから抽出されます。

In [ ]:
# グラウンディングチャンクには YouTube の直接リンクではなく
# Google のリダイレクト URL が含まれる。リダイレクトをたどって動画 ID を抽出。
YT_VIDEO_RE = re.compile(
    r"(?:youtube\.com/"
    r"(?:watch\?.*v=|shorts/|embed/|live/|v/)"
    r"|youtu\.be/)"
    r"([A-Za-z0-9_-]{11})"
)


def resolve_youtube_id(redirect_uri):
    """リダイレクト URL をたどり、YouTube 動画 ID を抽出する。"""
    try:
        resp = http_requests.head(
            redirect_uri, allow_redirects=True, timeout=5,
        )
        m = YT_VIDEO_RE.search(resp.url)
        return m.group(1) if m else ""
    except Exception:
        return ""


# REST で動画寄りのクエリを送信
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "ベラージオの噴水 ラスベガス "
                "YouTube 動画"
            ),
        }],
    }],
    "tools": [{"googleSearch": {}}],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

parts = ["### 動画検索結果\n\n"]
found = 0
for chunk in gm.get("groundingChunks", []):
    web = chunk.get("web", {})
    if not web:
        continue
    uri = web.get("uri", "")
    title = web.get("title", "")
    vid = resolve_youtube_id(uri)
    if vid:
        parts.append(f"**{title}**\n")
        parts.append(f"- 動画 ID: `{vid}`\n")
        yt_url = f"https://www.youtube.com/watch?v={vid}"
        parts.append(f"- URL: {yt_url}\n")
        thumb = (
            "https://i.ytimg.com"
            f"/vi/{vid}/hqdefault.jpg"
        )
        parts.append(f"- サムネイル: ![]({thumb})\n\n")
        found += 1
    if found >= 3:
        break

if found == 0:
    parts.append(
        "*グラウンディングチャンクに YouTube 動画が"
        "見つかりませんでした。ウェブ検索結果:*\n\n"
    )
    for chunk in gm.get("groundingChunks", [])[:5]:
        web = chunk.get("web", {})
        if web:
            t = web.get("title", "")
            u = web.get("uri", "")
            parts.append(f"- [{t}]({u})\n")

display(Markdown("".join(parts)))

---
## 6. 画像生成のための画像検索

これはセクション 4 のサムネイル画像検索とは異なる機能です。ここでは、`googleSearch.searchTypes.imageSearch` により、画像生成モデルが実際の Google 画像検索結果を画像生成時の**視覚的コンテキスト**として使用できます。

Concierge AI アプリでは、`request_postcard` がこの機能を使用して、実際のウェブ上の写真に基づいた、場所のフォトリアリスティックなセルフィーポストカードを生成します。

**重要:** プロンプトで明示的に検索を促す必要があります（例：「Search Google Images for photos of X, then generate...」）。これがないと、モデルは空の `groundingMetadata` で画像を生成します。

**注意:** これは `gemini-3.1-flash-image-preview`（プレビューモデル）を使用します。

In [ ]:
# 検索グラウンディング付きのネイティブ画像生成に対応したモデル
IMAGE_GEN_MODEL = "gemini-3.1-flash-image-preview"

# REST API — SDK はまだ searchTypes.imageSearch に未対応
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "Google 画像検索でラスベガスのベラージオの噴水の"
                "写真を検索し、それらの参考写真を使って、"
                "夕焼け時の噴水を鮮やかな色彩の美しい"
                "水彩画イラストとして生成してください。"
            ),
        }],
    }],
    "tools": [{
        "googleSearch": {
            # imageSearch は生成時の視覚的コンテキストを提供、
            # webSearch はウェブページからのテキストコンテキストを追加
            "searchTypes": {
                "imageSearch": {},
                "webSearch": {},
            },
        },
    }],
    "generationConfig": {
        # IMAGE を含めてネイティブ画像生成を有効化
        "responseModalities": ["TEXT", "IMAGE"],
    },
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(IMAGE_GEN_MODEL),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=120,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]

# 生成された画像を表示
for part in candidate.get("content", {}).get("parts", []):
    inline = (
        part.get("inlineData")
        or part.get("inline_data")
    )
    if inline and inline.get("data"):
        mime = (
            inline.get("mimeType")
            or inline.get("mime_type")
            or "image/png"
        )
        fmt = mime.split("/")[-1]
        display(IPImage(
            data=base64.b64decode(inline["data"]),
            format=fmt,
        ))
    if "text" in part:
        display(Markdown(part["text"]))

# 画像検索ソースを表示（帰属表示）
gm = candidate.get("groundingMetadata", {})
sources = []
for chunk in gm.get("groundingChunks", []):
    img_chunk = chunk.get("image") or {}
    web_chunk = chunk.get("web") or {}
    uri = (
        img_chunk.get("sourceUri")
        or web_chunk.get("uri")
        or ""
    )
    raw_title = (
        img_chunk.get("title")
        or web_chunk.get("title")
        or ""
    )
    title = re.sub(r"</?b>", "", raw_title).strip()
    if uri:
        sources.append(f"- [{title or uri}]({uri})")

if sources:
    display(Markdown(
        "### 画像ソース（帰属表示）\n\n"
        + "\n".join(sources[:5])
    ))
else:
    print(
        "画像検索ソースが返されませんでした"
        "（モデルが検索をトリガーしなかった可能性があります）。"
    )

---
## 7. ADK コンシェルジュエージェントの構築

ここまでのグラウンディング手法を [Agent Development Kit (ADK)](https://google.github.io/adk-docs/) を使って1つのコンシェルジュエージェントにまとめます。組み込みツール1つとカスタムツール3つを定義し、ADK エージェントに組み込みます。

カスタムツールは**サイドカーパターン**を使用します。適切なグラウンディングツール（`GoogleMaps`）を指定した別の `generateContent` 呼び出しか、Vector Search 2.0 API を直接呼び出し、構造化された結果を返します。ADK エージェントのメインモデルがツールの選択と結果の統合を行います。

### 7.1 Google 検索ツール（組み込み）

ADK には組み込みの `GoogleSearchTool` が用意されており、サイドカー API 呼び出しやカスタム関数なしで、モデルが生成中に直接 Google 検索を実行できます。モデルの応答は自動的にインライン引用付きで Google 検索結果にグラウンディングされます。

`bypass_multi_tools_limit=True` を設定すると、同じエージェント内でカスタム関数ツールと併用できます。このフラグがないと、単独のツールとしてのみ使用可能です。

In [ ]:
from google.adk.tools.google_search_tool import GoogleSearchTool

google_search_tool = GoogleSearchTool(
    bypass_multi_tools_limit=True,
)
print("google_search_tool 準備完了（組み込み）")

### 7.2 マップ検索ツール（カスタム）

`GoogleMaps` グラウンディングで `generateContent` を呼び出すサイドカーツール。`lat_lng` パラメータでラスベガス・ストリップ周辺に結果をバイアスします。

ADK は Python の関数シグネチャと Google スタイルのドキュメント文字列（`Args:` / `Returns:`）からツールの関数宣言を自動生成します。

In [ ]:
def maps_search(query: str) -> dict:
    """Google マップで周辺の場所を検索する。

    レストラン、観光地など、ラスベガス周辺のスポットを
    検索する場合に使用。

    Args:
        query: 検索内容。例: "ラスベガス・ストリップ
            周辺の寿司レストラン"。

    Returns:
        要約テキストと場所リストを含む dict。
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=query,
        config=GenerateContentConfig(
            tools=[Tool(
                google_maps=GoogleMaps(enable_widget=True)
            )],
            tool_config=ToolConfig(
                retrieval_config=RetrievalConfig(
                    lat_lng=LatLng(
                        latitude=36.1699,
                        longitude=-115.1398,
                    )
                ),
            ),
        ),
    )
    candidate = response.candidates[0] if response.candidates else None
    gm = getattr(candidate, "grounding_metadata", None)

    places = []
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            if chunk.maps:
                places.append({
                    "title": chunk.maps.title or "",
                    "uri": chunk.maps.uri or "",
                    "place_id": getattr(
                        chunk.maps, "place_id", ""
                    ),
                })

    return {
        "query": query,
        "summary": response.text or "",
        "places": places[:5],
    }


# テスト
print(maps_search("coffee near Las Vegas Strip")["places"][:2])

### 7.3 マップルートツール（カスタム）

経路案内用のサイドカーツール。単一のフリーテキストクエリではなく、`origin`、`destination`、`travel_mode` を別パラメータとして受け取り、Google マップグラウンディング用の明示的な経路案内プロンプトを構成します。これにより、「ホテル」のような曖昧な表現ではなく、構造化された住所がグラウンディング API に渡されます。

In [ ]:
def maps_route(
    origin: str,
    destination: str,
    travel_mode: str = "車",
) -> dict:
    """2地点間の経路案内・距離・所要時間を取得する。

    ゲストが行き方、所要時間、道順を聞いた場合に使用。

    Args:
        origin: 出発地のフル住所。例:
            "3950 S Las Vegas Blvd, Las Vegas, NV 89119"。
        destination: 目的地のフル名称または住所。例:
            "Red Rock Canyon"。
        travel_mode: "車"、"徒歩"、"自転車"、"公共交通機関"
            のいずれか。デフォルトは "車"。

    Returns:
        ルート概要と経由地を含む dict。
    """
    prompt = (
        f"{origin} から{destination}までの"
        f"{travel_mode}での行き方を教えてください。"
        f"合計の所要時間と距離も含めてください。"
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=GenerateContentConfig(
            tools=[Tool(
                google_maps=GoogleMaps(enable_widget=True)
            )],
        ),
    )
    candidate = response.candidates[0] if response.candidates else None
    gm = getattr(candidate, "grounding_metadata", None)

    places = []
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            if chunk.maps:
                places.append({
                    "title": chunk.maps.title or "",
                    "uri": chunk.maps.uri or "",
                    "place_id": getattr(
                        chunk.maps, "place_id", ""
                    ),
                })

    return {
        "origin": origin,
        "destination": destination,
        "travel_mode": travel_mode,
        "summary": response.text or "",
        "places": places[:5],
    }


# テスト
result = maps_route(
    "3950 S Las Vegas Blvd, Las Vegas, NV 89119",
    "Red Rock Canyon",
)
print(result["summary"][:200])

### 7.4 ホテル検索ツール（カスタム — Vector Search 2.0）

セクション 3 で作成した Vector Search 2.0 ホテルナレッジベースにクエリを送信するカスタムツール。`QUESTION_ANSWERING` タスクタイプでセマンティック検索を行い、MongoDB スタイルの `$eq` 演算子によるオプションのカテゴリフィルタリングをサポートします。

In [ ]:
def hotel_search(query: str, category: str = "") -> dict:
    """ホテルナレッジベースを検索する。

    客室、ダイニング、スパ、フィットネス、チェックイン・
    アウト、Wi-Fi、アクティビティ、空港からのアクセス、
    ポリシーなど、ホテルに関する質問に回答する場合に使用。

    Args:
        query: 自然言語の質問。例: "チェックアウトは何時"、
            "スパについて教えて"。
        category: 結果を絞り込むオプションのフィルター。
            "客室"、"ダイニング"、"スパ"、"フィットネス"、
            "アクティビティ"、"ポリシー"、"アクセス"、
            "Wi-Fi" のいずれか。

    Returns:
        該当するホテル情報アイテムを含む dict。
    """
    filter_kwargs = {}
    if category:
        filter_kwargs["filter"] = {
            "category": {"$eq": category}
        }

    request = vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COLLECTION_PATH,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="content_embedding",
            task_type="QUESTION_ANSWERING",
            top_k=3,
            output_fields=(
                vectorsearch_v1beta.OutputFields(
                    data_fields=["*"],
                )
            ),
            **filter_kwargs,
        ),
    )
    results = search_client.search_data_objects(request)

    items = []
    for item in results:
        data = item.data_object.data
        items.append({
            "title": data.get("title", ""),
            "content": data.get("content", "")[:300],
            "category": data.get("category", ""),
        })

    return {"query": query, "items": items}


# テスト
print(hotel_search("チェックアウトは何時")["items"][0]["title"])

### 7.5 コンシェルジュエージェントの作成とテスト

4つのツールをすべて1つの ADK エージェントに組み込みます。エージェントの指示（instruction）で各ツールの使い分けを教えます。`InMemoryRunner` を使うと、サーバーを立てずにノートブック上で簡単にテストできます。

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# ADK エージェントに Vertex AI 認証を使用（API キーではなく）
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

AGENT_INSTRUCTION = """
あなたはラスベガスの高級ホテルのコンシェルジュです。
ホテルの住所は 3950 S Las Vegas Blvd, Las Vegas, NV 89119 です。

Google 検索（グラウンディング）と3つのカスタムツールを使えます。

1. `hotel_search` — ホテル自体に関する質問（客室、ダイニング、スパ、
   フィットネス、チェックイン・アウト、Wi-Fi、アクティビティ、
   空港からのアクセス）。ホテル関連の質問にはまずこれを使ってください。
   オプションの `category` パラメータで結果を絞り込めます。

2. `maps_search` — 近くの場所やラスベガス周辺のスポットを検索する場合。

3. `maps_route` — 2地点間の経路案内・所要時間・距離を取得する場合。
   必ず市名・州名を含むフル住所を渡してください
   （例: "3950 S Las Vegas Blvd, Las Vegas, NV 89119"、
   「ホテル」ではなく）。出発地が指定されていない場合は
   ホテルの住所を起点として使用してください。

最新のイベント、ショーのスケジュール、チケット価格、観光地など、
ホテルのナレッジベース外の情報については、Google 検索グラウンディング
を使ってウェブから最新情報を取得してください。

常に親切で丁寧な口調で回答してください。ツールの結果から具体的な
情報（料金、時間、距離）を含めてください。
"""

concierge_agent = Agent(
    model=AGENT_MODEL_ID,
    name="concierge_agent",
    instruction=AGENT_INSTRUCTION,
    tools=[
        google_search_tool,
        hotel_search,
        maps_search,
        maps_route,
    ],
)

runner = InMemoryRunner(
    agent=concierge_agent, app_name="concierge_agent"
)

print("コンシェルジュエージェントの初期化が完了しました！")

さまざまな種類の質問でテストします。エージェントが各クエリに適切なツールを選択できることを確認します。

In [ ]:
# テスト: ホテル情報 → hotel_search (VS2) を呼ぶはず
events = await runner.run_debug(
    "チェックアウトは何時ですか？レイトチェックアウトはできますか？",
    quiet=True,
)
final = [e for e in events if e.is_final_response()]
if final:
    display(Markdown(final[-1].content.parts[0].text))

In [ ]:
# テスト: 最新イベント → 組み込み Google 検索を使うはず
events = await runner.run_debug(
    "今週ラスベガスで上演中のショーは何ですか？",
    quiet=True,
)
final = [e for e in events if e.is_final_response()]
if final:
    display(Markdown(final[-1].content.parts[0].text))

In [ ]:
# テスト: 周辺スポット → maps_search を呼ぶはず
events = await runner.run_debug(
    "ホテル近くのおすすめ寿司レストランを教えてください",
    quiet=True,
)
final = [e for e in events if e.is_final_response()]
if final:
    display(Markdown(final[-1].content.parts[0].text))

In [ ]:
# テスト: 経路案内 → maps_route を呼ぶはず
events = await runner.run_debug(
    "3950 S Las Vegas Blvd からレッドロックキャニオンまでの車での行き方は？",
    quiet=True,
)
final = [e for e in events if e.is_final_response()]
if final:
    display(Markdown(final[-1].content.parts[0].text))

---
## クリーンアップ

このノートブックで作成した Vector Search 2.0 コレクションを削除します。

In [ ]:
# デモコレクションを削除
try:
    item_ids = [item["id"] for item in HOTEL_INFO]
    reqs = [
        vectorsearch_v1beta.DeleteDataObjectRequest(
            name=(
                f"{COLLECTION_PATH}"
                f"/dataObjects/{obj_id}"
            )
        )
        for obj_id in item_ids
    ]
    data_client.batch_delete_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchDeleteDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=reqs,
            )
        )
    )
    print(f"{len(item_ids)} 件のデータオブジェクトを削除しました")

    op = vs_client.delete_collection(
        request=(
            vectorsearch_v1beta
            .DeleteCollectionRequest(
                name=COLLECTION_PATH,
            )
        )
    )
    op.result(timeout=300)
    print(f"コレクション '{COLLECTION_ID}' を削除しました")
except Exception as e:
    print(f"クリーンアップエラー: {e}")

継続的な課金を避けるために：

1. 不要になった場合は、Google Cloud プロジェクトを削除してください
2. または [Agent Platform](https://console.cloud.google.com/apis/api/aiplatform.googleapis.com) と [Vector Search](https://console.cloud.google.com/apis/api/vectorsearch.googleapis.com) API を無効にしてください